# EasyRAG Embedding Inputs And Provider Behavior

## Chapter position

This chapter turns prepared documents into retrieval infrastructure. Chunk boundaries, embedding inputs, vector storage, and workspace artifacts all start here.

## Learning question

How are embedding inputs assembled, and which parts change when you swap to provider-backed embeddings?

## Success criteria

- compare plain-text and multimodal embedding inputs
- inspect how EasyRAG builds provider-facing payloads from chunk metadata
- show how provider behavior changes what the vector backend can index

## Source code anchors

- `easyrag.rag.indexing.pipeline._build_embedding_input`
- `easyrag.rag.providers.default_embedding_func`
- `easyrag.config.models.get_embedding_model_name`
- `easyrag.config.models.get_embedding_base_url`


## Method principles

- `easyrag.rag.indexing.pipeline._build_embedding_input`: This helper builds the object that actually gets embedded. It is where plain text can stay plain text, while multimodal chunks can keep image paths attached for vision-language models.
- `easyrag.rag.providers.default_embedding_func`: This is the provider-backed embedding callable used when OpenAI-compatible configuration is available. It is the production-side replacement for the deterministic embedding stubs.
- `easyrag.config.models.get_embedding_model_name`: This config helper resolves which embedding model name the provider-backed path should call. It keeps deployment configuration outside the notebooks themselves.
- `easyrag.config.models.get_embedding_base_url`: This config helper resolves which provider base URL the embedding client should talk to. It separates notebook logic from environment-specific endpoint wiring.

Design reason: these anchors sit exactly where canonical documents fan out into chunks, vectors, graph records, and workspace files. The notebook uses them to make indexing inspectable enough that later retrieval behavior can be explained rather than guessed.


## How the code fits together

The flow in this notebook is chunk content -> embedding input builder -> provider boundary -> inspection. The goal is not to show every internal detail at once. The goal is to keep the boundary for this stage visible enough that later behavior still feels explainable. If a result looks odd, the intermediate objects in this notebook should tell you where to look next.

Design reason: the sequence moves from input corpus to persisted artifacts before any retrieval interpretation. That ordering keeps causality visible: if a later hit looks weak, you can still trace it back to chunking, normalization, or storage decisions made here.


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.config import get_embedding_base_url, get_embedding_model_name
from easyrag.rag.indexing.pipeline import build_insert_payloads
from easyrag.support.optional_deps import Document

## What this cell is proving

We first inspect the input contract itself. One document is plain text. One document carries `image_paths`, which makes the embedding input multimodal even before any provider is called.

Design reason: this cell performs the real indexing-side stage transition. Calling the helper directly keeps visible where raw inputs become canonical documents, chunks, vectors, or workspace records, which is exactly the boundary you need to inspect when later retrieval looks surprising.


In [ ]:
input_tmp = tempfile.TemporaryDirectory()
image_path = Path(input_tmp.name) / "diagram.png"
image_path.write_bytes(b"PNGDATA")

plain_document = Document(
    page_content="# Retrieval\nEasyRAG uses grounded retrieval and query rewrite.",
    metadata={
        "doc_id": "doc::plain",
        "path": "docs/retrieval.md",
        "relative_path": "docs/retrieval.md",
        "title": "retrieval",
        "source_type": "doc",
    },
)
multimodal_document = Document(
    page_content="Scanned PDF page 1 from manual.",
    metadata={
        "doc_id": "doc::pdf::page::1",
        "path": "docs/manual.pdf",
        "relative_path": "docs/manual.pdf",
        "title": "manual",
        "source_type": "pdf",
        "page_number": 1,
        "image_paths": [str(image_path)],
        "has_visual_content": True,
    },
)
input_rag = EasyRAG(
    working_dir=input_tmp.name,
    workspace="embedding-inputs-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
payloads = build_insert_payloads(input_rag, [plain_document, multimodal_document])
_print_json(payloads["vector_chunks"])

## Why this output looks like this

The plain-text chunk produces a string embedding input. The PDF-like chunk produces a dictionary with text plus image paths. That difference is what later lets a multimodal provider preserve page images instead of throwing them away.

Design reason: the output is concrete by design because architecture and pipeline claims are easy to overstate. Printing the actual artifact view keeps the abstraction boundary visible enough that later operational choices remain explainable.


In [ ]:
provider_summary = {
    "embedding_model_name": get_embedding_model_name(),
    "embedding_base_url": get_embedding_base_url(),
    "plain_input_type": type(payloads["vector_chunks"][0]["embedding_input"]).__name__,
    "multimodal_input_type": type(
        payloads["vector_chunks"][1]["embedding_input"]
    ).__name__,
}
_print_json(provider_summary)

## What this cell is proving

The provider-backed path should preserve the same contract even when the backend behavior is less deterministic.

If the environment is configured, the next cell sends the same inputs to the repo's default embedding function. The notebook only inspects the returned shape because provider-specific values vary.

In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_rag = EasyRAG(workspace="provider-embedding-inputs-demo")
    try:
        provider_vectors = provider_rag.embedding_func(
            [item["embedding_input"] for item in payloads["vector_chunks"]]
        )
        _print_json({"vector_lengths": [len(vector) for vector in provider_vectors]})
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")

## Common mistakes / debugging cues

- Do not tune retrieval before you understand chunk boundaries, embedding inputs, and backend choice.
- Watch metadata such as `chunk_strategy`, `vector_backend`, and workspace artifacts, not only final query hits.
- Indexing bugs often look like retrieval bugs until you inspect the payloads that were actually written.

## Takeaway

The key change is not the vector values themselves. It is the input object contract. Once `image_paths` survive into the embedding input, a multimodal provider can use visual context that a plain text-only provider would never see.

In [ ]:
input_tmp.cleanup()
print("Cleaned up the temporary embedding-input workspace.")

## Where to go next

- Continue with [03_06_vector_index_basics.ipynb](03_06_vector_index_basics.ipynb) to see how those embedding outputs become searchable records.
- Read [03-indexing-overview.md](../../docs/03-indexing-overview.md) for the stage-level indexing map.
- Revisit [02_03_pdf_and_multimodal_loading.ipynb](../02_data_loading/02_03_pdf_and_multimodal_loading.ipynb) if you want the loading-side view of these image paths.